In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')

import os
import tensorflow as tf
import keras
import cv2
import glob
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model
from tensorflow.keras import layers , models, optimizers

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.applications import ResNet50V2


# **Data Generator**

In [2]:
# specifing new image shape for resnet
img_shape = 224
batch_size = 64
train_data_path = './train/'
test_data_path = './test/'

In [3]:
train_preprocessor = ImageDataGenerator(
        rescale = 1 / 255.,
        rotation_range=10,
        zoom_range=0.2,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,                                        
        fill_mode='nearest',
        validation_split=0.25) # set validation split
    


test_preprocessor = ImageDataGenerator(
    rescale = 1 / 255.,
)

train_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode='rgb',
    batch_size=batch_size,
    subset='training', 
)


val_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode='rgb',
    batch_size=batch_size,
    subset='validation',
) # set as validation data


test_data = test_preprocessor.flow_from_directory(
    test_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

Found 21534 images belonging to 7 classes.
Found 7175 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


# **Calculating weights**

In [8]:


def generate_class_weights(class_series, multi_class=True, one_hot_encoded=False):
    """
    Generate class weights given a set of multi-class or multi-label labels, both one-hot-encoded or not.

    Input:
        class_series, list, list with the classes
        multi_class, boolean, indicates whether the input is multiclass or not
        one_hot_encoded, boolean, indicates whether the input is one hot encoder or not

        Examples of different formats of class_series:
        - ['a', 'b', 'c', 'd']
        - [[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 0, 0]]
        - [[0, 1, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0]]

    Output:
        dictionary, with the format { class_label: class_weight}

    """
    
    if multi_class:
        # If class is one hot encoded, transform to categorical labels to use compute_class_weight   
        if one_hot_encoded:
          class_series = np.argmax(class_series, axis=1)
      
        # Compute class weights with sklearn method
        class_labels = np.unique(class_series)
        class_weights = compute_class_weight(class_weight='balanced', classes=class_labels, y=class_series)
        return dict(zip(class_labels, class_weights))
    else:
        # It is neccessary that the multi-label values are one-hot encoded
        mlb = None
        if not one_hot_encoded:
          mlb = MultiLabelBinarizer()
          class_series = mlb.fit_transform(class_series)
    
        n_samples = len(class_series)
        n_classes = len(class_series[0])
    
        # Count each class frequency
        class_count = [0] * n_classes
        for classes in class_series:
            for index in range(n_classes):
                if classes[index] != 0:
                    class_count[index] += 1
        
        # Compute class weights using balanced method
        class_weights = [n_samples / (n_classes * freq) if freq > 0 else 1 for freq in class_count]
        class_labels = range(len(class_weights)) if mlb is None else mlb.classes_
          
        return dict(zip(class_labels, class_weights))



In [9]:
train_size = 21535

# One only batch
train_data_cat = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape,img_shape),
    color_mode='rgb',
    shuffle=True,
    batch_size=train_size,
    subset='training', 
)

for _, train_target in train_data_cat:
    break

pesos = generate_class_weights(train_target, multi_class=False, one_hot_encoded=True)
print(pesos)

Found 21534 images belonging to 7 classes.
{0: 1.0264550264550265, 1: 9.407601572739187, 2: 1.0010692203988656, 3: 0.5685244343533015, 4: 0.8260702777351542, 5: 0.8490990102913923, 6: 1.2931003422806702}


# **Fine-Tuning ResNet50V2**

In [11]:
ResNet50V2 = tf.keras.applications.ResNet50V2(input_shape=(224, 224, 3),
                                               include_top= False,
                                               weights='imagenet'
                                               )

In [12]:
# Freezing all layers except last 50

ResNet50V2.trainable = True

for layer in ResNet50V2.layers[:-50]:
    layer.trainable = False

In [13]:
def Create_ResNet50V2_Model():

    model = Sequential([
                      ResNet50V2,
                      Dropout(0.25),
                      BatchNormalization(),
                      Flatten(),
                      Dense(64, activation='relu'),
                      BatchNormalization(),
                      Dropout(0.5),
                      Dense(7,activation='softmax')
                    ])
    return model

In [14]:
ResNet50V2_Model = Create_ResNet50V2_Model()

ResNet50V2_Model.summary()

ResNet50V2_Model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50v2 (Functional)     (None, 7, 7, 2048)        23564800  
                                                                 
 dropout (Dropout)           (None, 7, 7, 2048)        0         
                                                                 
 batch_normalization (Batch  (None, 7, 7, 2048)        8192      
 Normalization)                                                  
                                                                 
 flatten (Flatten)           (None, 100352)            0         
                                                                 
 dense (Dense)               (None, 64)                6422592   
                                                                 
 batch_normalization_1 (Bat  (None, 64)                256       
 chNormalization)                                       

**Specifying Callbacks**

In [15]:
# Create Callback Checkpoint
checkpoint_path = "ResNet50V2_Model_Checkpoint_weight.h5"

Checkpoint = ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True)

# Create Early Stopping Callback to monitor the accuracy
Early_Stopping = EarlyStopping(monitor = 'val_accuracy', patience = 7, restore_best_weights = True, verbose=1)

# Create ReduceLROnPlateau Callback to reduce overfitting by decreasing learning
Reducing_LR = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',
                                                  factor=0.2,
                                                  patience=2,
#                                                   min_lr=0.00005,
                                                  verbose=1)

callbacks = [Checkpoint, Early_Stopping, Reducing_LR]

steps_per_epoch = train_data.n // train_data.batch_size
validation_steps = val_data.n // val_data.batch_size

In [16]:
history = ResNet50V2_Model.fit(train_data,
                               validation_data = val_data , 
                               epochs=20, 
                               batch_size=batch_size,
                               callbacks = callbacks, 
                               steps_per_epoch=steps_per_epoch, 
                               validation_steps=validation_steps,
                               class_weight=pesos
                              )

Epoch 1/20
336/336 [==============================] - ETA: 0s - loss: 1.7701 - accuracy: 0.3443

/opt/homebrew/lib/python3.11/site-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


336/336 [==============================] - 761s 2s/step - loss: 1.7701 - accuracy: 0.3443 - val_loss: 1.6107 - val_accuracy: 0.4231 - lr: 0.0010
Epoch 2/20
336/336 [==============================] - 775s 2s/step - loss: 1.5098 - accuracy: 0.4334 - val_loss: 2.1864 - val_accuracy: 0.4132 - lr: 0.0010
Epoch 3/20
336/336 [==============================] - 762s 2s/step - loss: 1.4431 - accuracy: 0.4634 - val_loss: 1.3413 - val_accuracy: 0.5033 - lr: 0.0010
Epoch 4/20
336/336 [==============================] - 754s 2s/step - loss: 1.3125 - accuracy: 0.5066 - val_loss: 1.3015 - val_accuracy: 0.5195 - lr: 0.0010
Epoch 5/20
336/336 [==============================] - 739s 2s/step - loss: 1.3058 - accuracy: 0.5124 - val_loss: 1.9126 - val_accuracy: 0.4665 - lr: 0.0010
Epoch 6/20
336/336 [==============================] - 756s 2s/step - loss: 1.2664 - accuracy: 0.5166 - val_loss: 1.2152 - val_accuracy: 0.5405 - lr: 0.0010
Epoch 7/20
336/336 [==============================] - 736s 2s/step - loss: 

In [ ]:
#ResNet50V2_Model = models.load_model("ResNet50V2_Model_Checkpoint_weight.h5")

# **Evaluating ResNet50V2**

In [18]:
ResNet50V2_Score = ResNet50V2_Model.evaluate(test_data)

print("    Test Loss: {:.5f}".format(ResNet50V2_Score[0]))
print("Test Accuracy: {:.2f}%".format(ResNet50V2_Score[1] * 100))

113/113 [==============================] - 181s 2s/step - loss: 0.9795 - accuracy: 0.6417
    Test Loss: 0.97951
Test Accuracy: 64.17%
